# Sentinel-1 · Pechea
Ruleaza celulele in ordine. Schimba `PROJECT_ID` cu ID-ul proiectului Google Cloud activat pentru Earth Engine. Ultima celula porneste exporturile doar daca setezi `PORNESTE_EXPORTUL = True`.

In [ ]:
%pip -q install earthengine-api geemap
import ee
from IPython.display import display

PROJECT_ID = "ID_PROIECT"
if PROJECT_ID == "ID_PROIECT":
    raise ValueError("Completeaza PROJECT_ID cu ID-ul proiectului tau Google Cloud.")

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)


## Functiile folosite
Codul pentru scene, comparatie, harta si export e inclus aici, ca sa poti rula notebook-ul direct.

In [ ]:
import ee


# Ordine: longitudine vest, latitudine sud, longitudine est, latitudine nord.
PECHEA_BOUNDS = [27.61, 45.46, 27.87, 45.74]

# Ultima zi din fiecare interval e exclusa. Datele sunt in UTC.
BEFORE = ("2024-08-15", "2024-09-14")
DURING = ("2024-09-14", "2024-09-19")
AFTER = ("2024-09-19", "2024-10-04")

# UTM 35N foloseste metri si e potrivit pentru Pechea.
CRS = "EPSG:32635"
PIXEL_SIZE_M = 10


# 1. Cautam scenele pentru zona si perioada pe care le-am ales.
def cauta_scene(aoi, perioada):
    return (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi)  # cauta scene care se intersecteaza cu zona
        .filterDate(*perioada)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.eq("resolution_meters", 10))  # distanta intre pixeli
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .sort("system:time_start")
    )


# 2. Afisam ce am gasit, ca sa putem alege scena dupa data si orbita.
def afiseaza_scene(scene):
    def adauga_detalii(imagine):
        detalii = ee.Dictionary({
            "id": imagine.get("system:index"),
            "data_utc": imagine.date().format("yyyy-MM-dd HH:mm:ss", "UTC"),
            "orbita": imagine.get("relativeOrbitNumber_start"),
            "directie": imagine.get("orbitProperties_pass"),
            "polarizari": imagine.get("transmitterReceiverPolarisation"),
        })
        return imagine.set("detalii", detalii)

    # getInfo aduce in Python doar lista de metadate, nu pixelii.
    detalii = scene.map(adauga_detalii).aggregate_array("detalii").getInfo()
    for rand in detalii:
        print(rand)
    return detalii


# 3. Alegem scena evenimentului si cautam scene din aceeasi geometrie.
def alege_scene(aoi, inainte, in_timpul, dupa, *, id_eveniment=None):
    def acopera_zona(scene):
        # Aici cerem ca amprenta scenei sa contina toata zona.
        # Verificarea pixelilor valizi se face separat, la comparatie.
        return scene.filter(
            ee.Filter.contains(leftField=".geo", rightValue=aoi, maxError=1)
        )

    def prima_sau_none(scene):
        if scene.limit(1).size().getInfo() == 0:
            return None
        return ee.Image(scene.first())

    candidate = acopera_zona(cauta_scene(aoi, in_timpul))
    if id_eveniment is not None:
        candidate = candidate.filter(ee.Filter.eq("system:index", id_eveniment))

    # Daca nu dam un ID, luam prima scena ca punct de plecare.
    eveniment = prima_sau_none(candidate)
    if eveniment is None:
        raise ValueError("Nu am gasit o scena de eveniment care acopera zona.")

    def aceeasi_geometrie(perioada):
        return (
            acopera_zona(cauta_scene(aoi, perioada))
            .filter(ee.Filter.eq(
                "relativeOrbitNumber_start", eveniment.get("relativeOrbitNumber_start")
            ))
            .filter(ee.Filter.eq(
                "orbitProperties_pass", eveniment.get("orbitProperties_pass")
            ))
        )

    # Inainte luam cea mai recenta scena. Dupa luam cea mai apropiata de eveniment.
    return {
        "before": prima_sau_none(aceeasi_geometrie(inainte).sort("system:time_start", False)),
        "during": eveniment,
        "after": prima_sau_none(aceeasi_geometrie(dupa).sort("system:time_start")),
    }


# 4. Extragem VV sau VH si punem imaginea pe grila comuna.
def banda_radar(imagine, aoi, banda="VV", *, pixel_m=PIXEL_SIZE_M, crs=CRS):
    if banda not in ("VV", "VH"):
        raise ValueError("Alege VV sau VH.")
    if imagine is None:
        raise ValueError("Lipseste scena pentru perioada aleasa.")
    if banda not in imagine.bandNames().getInfo():
        raise ValueError(f"Scena nu are banda {banda}.")

    # Valorile din S1_GRD sunt deja in dB, le pastram asa.
    # Folosim aceeasi origine si dimensiune a pixelului la toate scenele.
    grila = [pixel_m, 0, 0, 0, -pixel_m, 0]
    return imagine.select(banda).clip(aoi).reproject(
        crs=crs, crsTransform=grila
    ).clip(aoi)


# 5. Scadem valorile pixel cu pixel: scena noua minus scena de dinainte.
def calculeaza_diferenta(inainte_db, nou_db):
    # Exemplu: -20 - (-10) = -10 dB, deci semnalul a scazut.
    return nou_db.subtract(inainte_db).rename("diferenta_db")


# 6. Facem prima masca de apa nou aparuta.
def detecteaza_inundatia(nou_db, diferenta_db, *, prag_apa=-17, scadere_minima=3):
    if scadere_minima < 0:
        raise ValueError("Scaderea minima se da ca numar pozitiv.")

    # Pragurile sunt pentru test. Se aleg pe baza punctelor de referinta.
    # 1 = candidat inundat, 0 = nedetectat, pixel mascat = date lipsa.
    apa = nou_db.lt(prag_apa)
    a_scazut = diferenta_db.lt(-scadere_minima)
    return apa.And(a_scazut).rename("inundatie")


# 7. Vedem ce s-a schimbat intre doua masti facute fata de acelasi baseline.
def evolutia_inundatiei(masca_eveniment, masca_dupa):
    # 0 = nedetectat in ambele, 1 = retras, 2 = nou, 3 = persistent.
    # Pixelii fara date intr-una din masti raman mascati.
    return masca_eveniment.add(masca_dupa.multiply(2)).rename("evolutie")


# 8. Afisam straturile in notebook, cu aceeasi scala de gri pentru VV.
def afiseaza_harta(aoi, inainte_db, eveniment_db, diferenta_db, masca, evolutie=None):
    import geemap

    harta = geemap.Map()
    harta.centerObject(aoi, 12)
    gri = {"min": -25, "max": 0}  # doar pentru afisare
    harta.addLayer(inainte_db, gri, "VV inainte", False)
    harta.addLayer(eveniment_db, gri, "VV in timpul inundatiei")
    harta.addLayer(
        diferenta_db, {"min": -10, "max": 10, "palette": ["blue", "white", "red"]},
        "Diferenta in dB", False,
    )
    harta.addLayer(masca.selfMask(), {"palette": ["00e5ff"]}, "Candidati inundati")
    if evolutie is not None:
        harta.addLayer(
            evolutie.selfMask(),
            {"min": 1, "max": 3, "palette": ["orange", "blue", "purple"]},
            "Evolutie: retras / nou / persistent", False,
        )
    return harta


# 9. Exportam o banda sau o masca in Google Drive, ca GeoTIFF.
def exporta_geotiff(imagine, aoi, nume, *, folder="ROSPIN_Pechea",
                     pixel_m=PIXEL_SIZE_M, crs=CRS):
    # Pastram aceeasi grila la export. -9999 reprezinta pixelii fara date.
    nodata = -9999
    task = ee.batch.Export.image.toDrive(
        image=imagine.clip(aoi).toFloat().unmask(nodata, sameFootprint=False),
        description=nume,
        fileNamePrefix=nume,
        folder=folder,
        region=aoi,
        crs=crs,
        crsTransform=[pixel_m, 0, 0, 0, -pixel_m, 0],
        maxPixels=100_000_000,
        fileFormat="GeoTIFF",
        formatOptions={"cloudOptimized": True, "noData": nodata},
    )
    task.start()
    return task



# 10. Transformam pixelii candidati in poligoane pentru harta din site.
def exporta_poligoane(masca, aoi, nume, *, folder="ROSPIN_Pechea"):
    # selfMask scoate clasa 0. Pixelii fara date erau deja mascati.
    # 4 vecini: doua zone care se ating doar intr-un colt raman separate.
    poligoane = masca.selfMask().toInt().reduceToVectors(
        geometry=aoi,
        crs=CRS,
        crsTransform=[PIXEL_SIZE_M, 0, 0, 0, -PIXEL_SIZE_M, 0],
        geometryType="polygon",
        eightConnected=False,
        labelProperty="inundatie",
        maxPixels=100_000_000,
        geometryInNativeProjection=False,
    )
    task = ee.batch.Export.table.toDrive(
        collection=poligoane,
        description=nume,
        folder=folder,
        fileNamePrefix=nume,
        fileFormat="GeoJSON",
    )
    task.start()
    return task


# 11. Salvam datele exacte ale scenelor pentru timeline-ul din aplicatie.
def detalii_scena(imagine):
    if imagine is None:
        return None
    return ee.Dictionary({
        "id": imagine.get("system:index"),
        "acquired_at_utc": imagine.date().format("yyyy-MM-dd'T'HH:mm:ss'Z'", "UTC"),
        "relative_orbit": imagine.get("relativeOrbitNumber_start"),
        "pass_direction": imagine.get("orbitProperties_pass"),
    }).getInfo()


## Verifica scenele disponibile
Verifica ora UTC si copiaza ID-ul scenei dorite din lista. Daca lasi ID-ul gol, folosim prima scena care acopera zona de lucru.

In [ ]:
aoi = ee.Geometry.Rectangle(PECHEA_BOUNDS)
candidate = afiseaza_scene(cauta_scene(aoi, DURING))


In [ ]:
ID_EVENIMENT = None  # pune ID-ul intre ghilimele daca vrei o anumita scena
PRAG_APA = -17       # prag VV in timpul evenimentului, in dB
SCADERE_MINIMA = 3  # scaderea minima fata de scena dinainte, in dB

scene = alege_scene(aoi, BEFORE, DURING, AFTER, id_eveniment=ID_EVENIMENT)
for perioada, imagine in scene.items():
    print(perioada, detalii_scena(imagine))

if scene["before"] is None:
    raise ValueError("Nu avem o scena dinainte comparabila. Verifica intervalul BEFORE si ID-ul ales.")

# Comparam VV cu VV, pe aceeasi grila. Pragurile sunt valori de test.
inainte = banda_radar(scene["before"], aoi)
eveniment = banda_radar(scene["during"], aoi)
diferenta = calculeaza_diferenta(inainte, eveniment)
praguri = {"prag_apa": PRAG_APA, "scadere_minima": SCADERE_MINIMA}
masca = detecteaza_inundatia(eveniment, diferenta, **praguri)

imagini = {
    "before_vv": inainte, "during_vv": eveniment,
    "change_db": diferenta, "flood_mask": masca,
}
evolutie = None
if scene["after"] is not None:
    dupa = banda_radar(scene["after"], aoi)
    masca_dupa = detecteaza_inundatia(
        dupa, calculeaza_diferenta(inainte, dupa), **praguri
    )
    evolutie = evolutia_inundatiei(masca, masca_dupa)
    imagini.update(after_vv=dupa, flood_evolution=evolutie)

display(afiseaza_harta(aoi, inainte, eveniment, diferenta, masca, evolutie))


## Export pentru aplicatie
Dupa ce verifici harta, seteaza `PORNESTE_EXPORTUL = True` si ruleaza celula o data. Earth Engine scrie rezultatele in Google Drive, in `ROSPIN_Pechea`. `summary.json` se descarca din Colab cu celula de la final.

In [ ]:
import json
from pathlib import Path

PORNESTE_EXPORTUL = False
if PORNESTE_EXPORTUL:
    sumar = {
        "event_id": "pechea",
        "name": "Pechea, Galati",
        "source": "COPERNICUS/S1_GRD",
        "aoi_bbox_wgs84": PECHEA_BOUNDS,
        "pixel_size_m": PIXEL_SIZE_M,
        "crs": CRS,
        "polarisation": "VV",
        "thresholds_db": praguri,
        "scenes": {perioada: detalii_scena(imagine) for perioada, imagine in scene.items()},
        "files": {**{nume: f"{nume}.tif" for nume in imagini},
                  "flood_extent": "flood_extent.geojson"},
        "mask_legend": {"0": "nedetectat", "1": "candidat inundat", "-9999": "fara date"},
    }
    taskuri = {nume: exporta_geotiff(imagine, aoi, nume)
               for nume, imagine in imagini.items()}
    taskuri["flood_extent"] = exporta_poligoane(masca, aoi, "flood_extent")
    Path("summary.json").write_text(json.dumps(sumar, indent=2), encoding="utf-8")
    print("Exporturi pornite. Verifica starea lor in celula urmatoare.")
else:
    print("Exportul e oprit. Seteaza PORNESTE_EXPORTUL = True cand harta e gata.")


In [ ]:
# Ruleaza din nou celula asta pentru starea exporturilor, fara sa le repornesti.
for nume, task in globals().get("taskuri", {}).items():
    status = task.status()
    print(nume, status.get("state"), status.get("error_message", ""))


In [ ]:
from google.colab import files
from pathlib import Path

if Path("summary.json").exists():
    files.download("summary.json")
else:
    print("Porneste intai exportul ca sa se genereze summary.json.")
